In [1]:
import fastmcp
from fastmcp import FastMCP
# fastmcp for creating and connecting MCP servers,clients, and hosts

In [44]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=2
    
)

In [3]:
from langchain_core.tools import tool

In [4]:
import socket
PORT = 8000

def test_port(port=PORT):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.bind(('127.0.0.1', port))
            return False
        except socket.error:
            return True

f"Port {PORT} is available: {not test_port()}"

'Port 8000 is available: True'

#### tools in langchain

In [5]:
@tool
def multiply(a:int,b:int)->int:
    """Multiply two number"""
    return a*b
print('='*80)
print("the names of the tools is:",multiply.name)
print("the descriptions of the tools:",multiply.description)
print("the arguments of the tools",multiply.args)
print('='*80)
print(f"product:{multiply.invoke({'a':2,'b':3})}")
print('='*80)

the names of the tools is: multiply
the descriptions of the tools: Multiply two number
the arguments of the tools {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
product:6


#### Creating a calculator MCP Server

In [6]:
# creating fastmcp server object
mcp=FastMCP(
    name="CalculatorMCPServer",
    instructions="""
    This server provides data analysis tools
    call get_average() to analyze numerical data
    """
)
print("mcp object",mcp)

mcp object FastMCP('CalculatorMCPServer')


#### tools
* we define the MCP tools `add` and `subtrct`
* these will be called on the MCP object

In [7]:
@mcp.tool
def add(a:int,b:int)->int:
    """Add two integers together
    Args:
    a (int): the first integers
    b (int): the second integers
    
    Returns:
    int: the sum of a and b
    
    Example:
    >>> add(3,5)
    8
    """
    return a+b

@mcp.tool
def subtract(a:int,b:int)-> int:
    """Subtract the two given numbers
    
    Args:
    a (int): first integer 
    b (int): the second integer
    
    Return:
    int : the sum of `a` and `b`.
    
    Example:
    >>> subtract(3,1)
    1
    """
    return a-b

#### Resources

In [8]:
@mcp.resource('file:///endpoint/{name}')
def return_template_document(name:str)->str:
    """Read a document by name"""
    with open(f"resources_path/{name}",'r') as f:
        return f.read()

In [9]:
import os 
def make_dirs():
    if os.path.exists("resources_path"):
        print("path directory already exist")
    else:
        os.makedirs("resources_path")
        print("path directory created")

In [10]:
make_dirs()

path directory already exist


When you call this resources, it will require the input `{name}` to identify which document to retrieve.`path/{name}` is where files actually exist on disk. The function reads from this physical file system location to return the content.

**Note:** The URI endpoint is the MCP address for requesting resources, while the path is the actual storage location on your system. They are related but distinct.


In [11]:
@mcp.resource("file://endpoint2/{name}")
def read_document(name:str)->str:
    """Read a document by name from the path directory"""
    try:
        with open(f"resources_path/{name}","r") as f:
            return f.read()
    except FileNotFoundError:
        return f"document {name} not found in the path directory"
    except Exception as e:
        return f"Error reading document:{str(e)}"
    

#### Prompts 

Prompts are consistent, reusable templates that can be called for simple, repetitive tasks. They capture domain expertise in a structured way, so instead of reinventing instructions each time, the AI can rely on a proven pattern.


In [12]:
@mcp.prompt(title="Code Review")
def review_code(code:str)->str:
    return f"please review this code:\n\n{code}"

#### Creating a Client: In-memory-transport

In [13]:
from fastmcp import Client
client=Client(mcp)
print(f"clinet:{client}")

clinet:<fastmcp.client.client.Client object at 0x000001BD9B825550>


#### Creating tools

In [14]:
async def call_add_tool(a:int,b:int):
    async with client:
        result=await client.call_tool("add",{"a":a,"b":b})
        return result

In [15]:
response=await call_add_tool(4,5)
print(response.content)

[TextContent(type='text', text='9', annotations=None, meta=None)]


In [16]:
# The actual answer/data
print("\nResult Data .data :")
print(response.data)  # 9

# Content (text format)
print("\nContent (as text):")
print(response.content[0].text)  # "9"

# Structured content (as dictionary)
print("\nStructured Content:")
print(response.structured_content)  


Result Data .data :
9

Content (as text):
9

Structured Content:
{'result': 9}


#### Fetching Available Tools

In [17]:
async with client:
    tools = await client.list_tools()
    print('='*80)
    print("Availabe tools:")
    for tool in tools:
        print('='*80)
        print("the name of tool is:",tool.name)
        print("the description of the tool is:",tool.description)

Availabe tools:
the name of tool is: add
the description of the tool is: Add two integers together
Args:
a (int): the first integers
b (int): the second integers

Returns:
int: the sum of a and b

Example:
>>> add(3,5)
8
the name of tool is: subtract
the description of the tool is: Subtract the two given numbers

Args:
a (int): first integer 
b (int): the second integer

Return:
int : the sum of `a` and `b`.

Example:
>>> subtract(3,1)
1


#### Fetching the resources

In [18]:
async def call_resources(name):
    async with client:
        result= await client.read_resource(f"file:///endpoint/{name}")
        return result
    

In [19]:
response=await call_resources("example.txt")
resource=response[0]


In [20]:
print(f"uri:      {resource.uri}")
print(f"mimeType: {resource.mimeType}")
print(f"meta:     {resource.meta}")
print(f"text:     {resource.text}")

uri:      file:///endpoint/example.txt
mimeType: text/plain
meta:     None
text:     hi, welcome to the mcp tutorial

this is introuductions about mcp 


In [21]:
async def call_resource_2(name):
    async with client:
        result= await client.read_resource(f"file://endpoint2/{name}")
        return result

In [22]:
response=await call_resource_2('readme.txt')
resource=response[0]

In [23]:
print(f"uri:      {resource.uri}")
print(f"mimeType: {resource.mimeType}")
print(f"meta:     {resource.meta}")
print(f"text:     {resource.text}")

uri:      file://endpoint2/readme.txt
mimeType: text/plain
meta:     None
text:     # Documents Folder

This folder contains documents accessible through the Calculator MCP Server.

## Available Documents:

- examples.txt - Examples of how to use the calculator
- README.txt - This file



#### Prompts
we'll also create a function to call the prompts method to review code. the only parameters is the code content to be reviewed

In [24]:
async def call_prompt(code):
    async with client:
        result=await client.get_prompt("review_code",{"code":code})
        return result

In [25]:
response = await call_prompt("CODE TO BE REVIEWED")

In [26]:
message=response.messages[0]
print(f"Prompt Role:{message.role}")
print(f"Prompt Content:{message.content.text}")

Prompt Role:user
Prompt Content:please review this code:

CODE TO BE REVIEWED


#### HTTP Transport MCP Servers

##### Starting HTTP MCP Server

In [ ]:
import asyncio
asyncio.create_task(mcp.run_http_async(port=PORT))
print(f"HTTP MCP Server started in background on port {PORT}")

HTTP MCP Server started in background on port 8000




┌─────────────────────────────────────────────────────────────────────────────┐
│                                                                             │
│                                                                             │
│                        ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                        █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                             │
│                                                                             │
│                                                                             │
│                                FastMCP 3.4.2                                │
│                            https://gofastmcp.com                            │
│                                                                             │
│                 🖥  Server:      CalculatorMCPServer, 3.4.2                  │
│                 🚀 Deploy free: https

[07/05/26 14:37:15] INFO     Starting MCP server 'CalculatorMCPServer' with transport 'http' on    transport.py:304
                             http://127.0.0.1:8000/mcp                                                             

INFO:     Started server process [9200]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:64196 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64197 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:64198 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64199 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64200 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64201 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64202 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64203 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:64204 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64205 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64206 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64230 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64231 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:64232 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64233 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64236 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58722 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58723 -

#### HTTP Transport and Client

##### HTTP Transport
Now we'll create an MCP client that uses `HTTP` transport to communicate with the server. This creates a transport object that tells the MCP Client how to communicate with the MCP server over HTTP.


In [28]:
from fastmcp.client.transports import StdioTransport,StreamableHttpTransport
transport_http=StreamableHttpTransport(url=f"http://127.0.0.1:{PORT}/mcp")

In [29]:
# lets create the client using transport http object
http_client = Client(transport_http)
print('http_client',http_client )


http_client <fastmcp.client.client.Client object at 0x000001BD9CA6D610>


In [30]:
async def test_client_http(client:Client,a:int,b:int)->int:
    async with client:
        result=await client.call_tool("add",{"a":a,"b":b})
        return result

In [31]:
response=await test_client_http(http_client,4,5)
print(response.content[0].text)

9


In [32]:
async with http_client:
    tools = await http_client.list_tools()
    print('='*80)
    print("Availabe tools:")
    for tool in tools:
        print('='*80)
        print("the name of tool is:",tool.name)
        print("the description of the tool is:",tool.description)

Availabe tools:
the name of tool is: add
the description of the tool is: Add two integers together
Args:
a (int): the first integers
b (int): the second integers

Returns:
int: the sum of a and b

Example:
>>> add(3,5)
8
the name of tool is: subtract
the description of the tool is: Subtract the two given numbers

Args:
a (int): first integer 
b (int): the second integer

Return:
int : the sum of `a` and `b`.

Example:
>>> subtract(3,1)
1


#### Langchain Tools With HTTP MCP Servers

Converting an MCP tool to a LangChain tool is relatively simple. The main challenge is that it requires a more complex client and transport layer — langchain-mcp-adapters. The ```ClientSession``` essentially acts as a client that retrieves information from ```StreamableHttpTransport```, which is a bit more complex.
Before we dive deeper into ```StreamableHttpTransport```, let’s import ```ClientSession```, ```load_mcp_tools```, ```create_react_agent```, and our LLM.


In [33]:
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain.agents import create_agent
from mcp import ClientSession

The ```streamablehttp_client``` function connects you to an MCP server over HTTP. When you call it, it opens a connection and gives you back three things that let you communicate with the server:

```read:``` Receives messages from the server

```write:``` Sends messages to the server

```_sid:``` Returns your unique connection ID


In [35]:
from pprint import pprint
from mcp.client.streamable_http import streamable_http_client
async with streamable_http_client(f"http://127.0.0.1:{PORT}/mcp") as (read, write, _sid):
    print('='*80)
    pprint(read)    
    print('='*80)
    pprint(write)
    print('='*80)
    print(_sid)


    

MemoryObjectReceiveStream(_state=_MemoryObjectStreamState(max_buffer_size=0,
                                                          buffer=deque([]),
                                                          open_send_channels=1,
                                                          open_receive_channels=1,
                                                          waiting_receivers=OrderedDict(),
                                                          waiting_senders=OrderedDict()),
                          _closed=False)
MemoryObjectSendStream(_state=_MemoryObjectStreamState(max_buffer_size=0,
                                                       buffer=deque([]),
                                                       open_send_channels=1,
                                                       open_receive_channels=1,
                                                       waiting_receivers=OrderedDict(),
                                                       waiting_senders

To use MCP tools with a LangChain agent, we first establish a connection and initialize the session. Then load_mcp_tools converts the MCP tools into LangChain-compatible format. We create the agent with our LLM and converted tools, then make a query. Everything stays wrapped in async context managers to keep the session alive while the agent we use ```ainvoke``` not  ```invoke``` for the agent
.

In [ ]:
async with streamable_http_client(f"http://127.0.0.1:{PORT}/mcp") as (read,write,_sid):
    async with ClientSession(read,write) as session:
        await session.initialize()
        
        # load langchain tools from the Live MCP session
        tools=await load_mcp_tools(session)
        
        # Build the agent while the sessions is still open
        agent=create_agent(
            model=llm,
            tools=tools,
        )
        agent_response=await agent.ainvoke({"message":"use the add tool to add 2 and 3 and let me know if you used a tool."})
    

In [ ]:
print(agent_response['messages'][-1].content)